# Post Selection RNA Sequence Network Analysis

Reproducible analysis of abundance-annotated FASTA sequences: **MAFFT alignment → nucleotide alignment map → pairwise distances → hierarchical clusters → cluster abundances → radial sequence network**.

**input file:** a FASTA file with a normalized relative abundance as the final whitespace-separated field in each header, e.g. `>example_sequence 0.1`. The values are interpreted as fractions (0.1 = 10%).

Run cells in order. Install MAFFT separately (e.g. `brew install mafft` on macOS); install Python dependencies with `pip install -r requirements.txt`. The notebook writes results to an `outputs/` directory.

## 1. Configuration and imports

In [ ]:
from pathlib import Path
import shutil
import subprocess
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from Bio import AlignIO, SeqIO
from matplotlib.colors import ListedColormap
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
from IPython.display import display

# Input file: change this filename to analyse another FASTA file.
input_file = Path("your_sequences.fasta")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
aligned_file = output_dir / "rna_sequence_aligned.fasta"

# Analysis parameters
cluster_cutoff = 10          # Average-linkage hierarchical clustering cutoff
network_threshold = 3       # Add an edge for genetic distance <= this value

# Optional minimum sequence length
# Set to None to disable length filtering
min_length = 50

# Radial network appearance
node_size_base = 1500
node_size_scale = 250000
label_font_size = 14
node_gap = 0.35
cluster_gap = 0.8
figure_size = (20, 16)

if not input_file.is_file():
    raise FileNotFoundError(f"input file not found: {input_file.resolve()}")
if shutil.which("mafft") is None:
    raise RuntimeError("MAFFT not found. Install it first (macOS: brew install mafft).")
print(f"input file: {input_file.resolve()}")

## 2. Read abundances and align sequences

MAFFT aligns the sequences; FASTA record identifiers are used to match abundances back to aligned records.

In [ ]:
records = list(SeqIO.parse(input_file, "fasta"))
if min_length is not None:
    records = [record for record in records if len(str(record.seq).replace("-", "").replace(".", "")) >= min_length]
if len(records) < 2:
    raise ValueError("input file must contain at least two sequences.")
if len({record.id for record in records}) != len(records):
    raise ValueError("FASTA sequence IDs must be unique.")
try:
    abundances = {record.id: float(record.description.split()[-1]) for record in records}
except ValueError as exc:
    raise ValueError("Each FASTA header must end with a numeric normalized abundance.") from exc
if not all(np.isfinite(v) and v >= 0 for v in abundances.values()):
    raise ValueError("Abundances must be finite and nonnegative.")

alignment_input = input_file
if min_length is not None:
    alignment_input = output_dir / "rna_sequence_filtered.fasta"
    SeqIO.write(records, alignment_input, "fasta")

with aligned_file.open("w") as output:
    subprocess.run(["mafft", "--auto", str(alignment_input)], stdout=output, check=True)
alignment = AlignIO.read(aligned_file, "fasta")
node_abundances = np.array([abundances[record.id] for record in alignment])
print(f"Sequences: {len(alignment)} | Aligned positions: {alignment.get_alignment_length()}")
print(f"Total abundance represented: {node_abundances.sum():.2%}")

## 3. Visualize the multiple sequence alignment

In [ ]:
mapping = {"A": 0, "C": 1, "G": 2, "T": 3, "U": 3, "-": 4}
seq_matrix = np.array([list(str(record.seq).upper()) for record in alignment])
numeric_matrix = np.array([[mapping.get(base, 4) for base in row] for row in seq_matrix])
cmap_alignment = ListedColormap(["#4daf4a", "#377eb8", "#ff7f00", "#e41a1c", "#eeeeee"])
fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(numeric_matrix, aspect="auto", interpolation="nearest", cmap=cmap_alignment, vmin=-0.5, vmax=4.5)
ax.set(xlabel="Aligned nucleotide position", ylabel="RNA sequence variant", title="Multiple sequence alignment of RNA sequences")
ax.set_yticks(range(len(alignment)), [record.id for record in alignment], fontsize=7)
fig.tight_layout()
fig.savefig(output_dir / "rna_sequence_alignment.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Pairwise genetic distances

**Distance definition:** count substitutions where both aligned sequences have a nucleotide, plus gap-versus-base positions between the first and last positions where both sequences have nucleotides. Terminal gaps are ignored. This is a descriptive difference count, **not an evolutionary distance estimate**; a distance of zero can occur for sequences with different terminal extensions.

In [ ]:
def genetic_distance(seq1, seq2):
    seq1, seq2 = str(seq1).upper(), str(seq2).upper()
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must have equal aligned lengths.")
    shared = [i for i, (a, b) in enumerate(zip(seq1, seq2)) if a != "-" and b != "-"]
    if not shared:
        return np.nan
    substitutions = sum(seq1[i] != seq2[i] for i in shared)
    internal_indels = sum((seq1[i] == "-") != (seq2[i] == "-") for i in range(shared[0], shared[-1] + 1))
    return substitutions + internal_indels

n_sequences = len(alignment)
distance_matrix = np.zeros((n_sequences, n_sequences))
for i in range(n_sequences):
    for j in range(i + 1, n_sequences):
        d = genetic_distance(alignment[i].seq, alignment[j].seq)
        distance_matrix[i, j] = distance_matrix[j, i] = d
if not np.isfinite(distance_matrix).all():
    raise ValueError("Some sequence pairs share no aligned nucleotides; clustering cannot proceed.")
pairwise_distances = distance_matrix[np.triu_indices(n_sequences, k=1)]
print(f"Mean: {pairwise_distances.mean():.2f} | Median: {np.median(pairwise_distances):.1f} | Min: {pairwise_distances.min():.0f} | Max: {pairwise_distances.max():.0f}")
pd.DataFrame(distance_matrix, index=[r.id for r in alignment], columns=[r.id for r in alignment]).to_csv(output_dir / "rna_sequence_distance_matrix.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(distance_matrix, cmap="viridis")
fig.colorbar(im, ax=axes[0], label="Genetic distance")
axes[0].set(xlabel="Sequence index", ylabel="Sequence index", title="Pairwise distances")
axes[1].hist(pairwise_distances, bins=20, edgecolor="black")
axes[1].set(xlabel="Genetic distance", ylabel="Number of sequence pairs", title="Distance distribution")
fig.tight_layout()
fig.savefig(output_dir / "rna_sequence_distance_summary.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Hierarchical clustering

Average linkage is applied to the precomputed distance matrix. `cluster_cutoff` controls cluster membership; it is independent of `network_threshold`, which controls visible edges.

In [ ]:
Z = linkage(squareform(distance_matrix), method="average")
clusters = fcluster(Z, t=cluster_cutoff, criterion="distance")
print(f"Number of clusters: {len(set(clusters))}")
for cluster, count in sorted(Counter(clusters).items()):
    print(f"Cluster {cluster}: {count} sequences")

fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(Z, labels=[record.id for record in alignment], leaf_rotation=90, leaf_font_size=8, color_threshold=cluster_cutoff, ax=ax)
ax.axhline(cluster_cutoff, color="red", linestyle="--", linewidth=1, label=f"Cutoff = {cluster_cutoff}")
ax.set(xlabel="RNA sequence variants", ylabel="Genetic distance", title="Hierarchical clustering of RNA sequences")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "rna_sequence_dendrogram.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Cluster abundance

Cluster abundance is the **sum of the input normalized abundances** for sequences in each cluster. If the input contains only a subset of variants, this is their contribution to the original population, not a renormalization to 100%.

In [ ]:
cluster_df = pd.DataFrame({
    "Sequence_ID": [record.id for record in alignment],
    "Sequence": [str(record.seq).replace("-", "") for record in alignment],
    "Cluster": clusters,
    "Abundance": node_abundances,
})
cluster_df.to_csv(output_dir / "rna_sequence_clusters.csv", index=False)
cluster_summary = (cluster_df.groupby("Cluster", as_index=False)
                   .agg(Sequence_count=("Sequence_ID", "size"), Total_abundance=("Abundance", "sum")))
top_sequences = (cluster_df.sort_values("Abundance", ascending=False)
                 .drop_duplicates("Cluster").set_index("Cluster")["Sequence_ID"])
cluster_summary["Top_Sequence"] = cluster_summary["Cluster"].map(top_sequences)
cluster_summary["Abundance_percent"] = 100 * cluster_summary["Total_abundance"]
cluster_summary = cluster_summary.sort_values("Total_abundance", ascending=False)
cluster_summary.to_csv(output_dir / "rna_sequence_cluster_abundance.csv", index=False)
display(cluster_summary)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(cluster_summary["Top_Sequence"], cluster_summary["Abundance_percent"])
ax.set(xlabel="Most abundant sequence in cluster", ylabel="Abundance (% of input population)", title="Relative abundance of RNA sequence clusters")
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
fig.savefig(output_dir / "rna_sequence_cluster_abundance.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Network connectivity sensitivity

Changing the edge threshold changes the number of edges and connected components, **not** the hierarchical cluster assignments.

In [ ]:
thresholds = list(range(1, 16))
component_counts, edge_counts = [], []
for t in thresholds:
    temp_graph = nx.Graph()
    temp_graph.add_nodes_from(range(n_sequences))
    temp_graph.add_edges_from((i, j) for i in range(n_sequences) for j in range(i + 1, n_sequences) if distance_matrix[i, j] <= t)
    component_counts.append(nx.number_connected_components(temp_graph))
    edge_counts.append(temp_graph.number_of_edges())
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, component_counts, marker="o", label="Connected components")
ax.plot(thresholds, edge_counts, marker="s", label="Edges")
ax.axvline(network_threshold, color="gray", linestyle="--", label=f"Chosen threshold = {network_threshold}")
ax.set(xlabel="Genetic distance edge threshold", ylabel="Count", title="Network connectivity across thresholds")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(output_dir / "rna_sequence_network_thresholds.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Abundance-weighted radial sequence network

Each cluster's highest-abundance sequence is central; other members are placed around a circle. **Node area** reflects abundance and **node colour** indicates hierarchical cluster. Edges join pairs with distance ≤ `network_threshold`, including pairs in different clusters. Radial and angular positions are for legibility, **not genetic distance**. The physical-size spacing heuristic is approximate; label collisions can still occur.

In [ ]:
G = nx.Graph()

for i, record in enumerate(alignment):
    G.add_node(
        i,
        label=record.id.split("/")[0],
        abundance=node_abundances[i],
        cluster=int(clusters[i])
    )

for i in range(len(alignment)):
    for j in range(i + 1, len(alignment)):
        d = distance_matrix[i, j]

        if np.isfinite(d) and d <= network_threshold:
            G.add_edge(i, j, weight=d)

# GROUP SEQUENCES BY CLUSTER

cluster_nodes = defaultdict(list)

for node in G.nodes():
    cluster_nodes[G.nodes[node]["cluster"]].append(node)

# IDENTIFY MOST ABUNDANT SEQUENCES


cluster_centres = {}

for cluster, nodes in cluster_nodes.items():
    centre = max(
        nodes,
        key=lambda n: G.nodes[n]["abundance"]
    )

    cluster_centres[cluster] = centre

    print(
        f"Cluster {cluster}: "
        f"{G.nodes[centre]['label']} "
        f"(abundance = {G.nodes[centre]['abundance']:.4%})"
    )

# CALCULATE NODE DIAMETERS

def plot_node_size(node):
    return (
        node_size_base
        + node_size_scale * G.nodes[node]["abundance"]
    )

def plot_node_diameter(node):
    # Approximate diameter in inches
    return 2 * np.sqrt(plot_node_size(node) / np.pi) / 72

node_diameters = {
    node: plot_node_diameter(node)
    for node in G.nodes()
}

# CALCULATE CLUSTER RADII

cluster_radii = {}

for cluster, nodes in cluster_nodes.items():
    centre = cluster_centres[cluster]
    others = [n for n in nodes if n != centre]

    if not others:
        cluster_radii[cluster] = node_diameters[centre] / 2
        continue

    # Required circumference for peripheral nodes
    required_circumference = sum(
        node_diameters[n] + node_gap
        for n in others
    )

    circumference_radius = (
        required_circumference / (2 * np.pi)
    )

    # Ensure the central node doesn't overlap
    centre_clearance = (
        node_diameters[centre] / 2
        + max(node_diameters[n] for n in others) / 2
        + node_gap
    )

    cluster_radii[cluster] = max(
        circumference_radius,
        centre_clearance
    )

# POSITION CLUSTERS COMPACTLY

sorted_clusters = sorted(
    cluster_nodes,
    key=lambda c: cluster_radii[c],
    reverse=True
)

cluster_positions = {}

# Largest cluster at the origin
largest = sorted_clusters[0]
cluster_positions[largest] = np.array([0.0, 0.0])

# Pack remaining clusters around the largest
for cluster in sorted_clusters[1:]:
    radius = cluster_radii[cluster]
    placed = False

    for search_radius in np.arange(0.5, 200, 0.25):
        for theta in np.linspace(
            0, 2 * np.pi, 180, endpoint=False
        ):
            candidate = np.array([
                search_radius * np.cos(theta),
                search_radius * np.sin(theta)
            ])

            collision = False

            for other, other_pos in cluster_positions.items():
                required_distance = (
                    radius
                    + cluster_radii[other]
                    + cluster_gap
                )

                if np.linalg.norm(candidate - other_pos) < required_distance:
                    collision = True
                    break

            if not collision:
                cluster_positions[cluster] = candidate
                placed = True
                break

        if placed:
            break

    if not placed:
        raise RuntimeError(
            f"Could not position cluster {cluster}"
        )

# POSITION NODES RADIALLY

pos = {}

for cluster, nodes in cluster_nodes.items():
    cx, cy = cluster_positions[cluster]

    centre = cluster_centres[cluster]
    pos[centre] = np.array([cx, cy])

    others = [n for n in nodes if n != centre]

    if not others:
        continue

    radius = cluster_radii[cluster]

    # Allocate angular space according to node size
    arc_lengths = np.array([
        node_diameters[n] + node_gap
        for n in others
    ])

    angles = (
        2 * np.pi
        * (np.cumsum(arc_lengths) - arc_lengths / 2)
        / arc_lengths.sum()
    )

    for node, theta in zip(others, angles):
        pos[node] = np.array([
            cx + radius * np.cos(theta),
            cy + radius * np.sin(theta)
        ])

# VISUALISE NETWORK

# Create exactly ONE figure
fig, ax = plt.subplots(figsize=figure_size)

unique_clusters = sorted(cluster_nodes.keys())
cmap = plt.get_cmap("tab10", len(unique_clusters))

cluster_colors = {
    cluster: cmap(i)
    for i, cluster in enumerate(unique_clusters)
}

# Draw edges
for u, v, data in G.edges(data=True):
    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=[(u, v)],
        width=5 / (1 + data["weight"]),
        alpha=0.45,
        edge_color="gray",
        ax=ax
    )

# Draw nodes
for cluster, nodes in cluster_nodes.items():
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=nodes,
        node_size=[plot_node_size(n) for n in nodes],
        node_color=[cluster_colors[cluster]],
        edgecolors="black",
        linewidths=1,
        alpha=0.9,
        ax=ax
    )

# Highlight most abundant sequences
for cluster, centre in cluster_centres.items():
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=[centre],
        node_size=plot_node_size(centre),
        node_color=[cluster_colors[cluster]],
        edgecolors="black",
        linewidths=3,
        ax=ax
    )

# Labels
labels = {
    node: G.nodes[node]["label"]
    for node in G.nodes()
}

nx.draw_networkx_labels(
    G,
    pos,
    labels=labels,
    font_size=label_font_size,
    font_weight="bold",
    ax=ax
)


ax.set_title("Abundance-weighted RNA sequence network\nMost abundant variant centred within each cluster", fontsize=18, pad=20)
ax.set_aspect("equal")
ax.axis("off")
ax.margins(0.08)
fig.tight_layout()
fig.savefig(output_dir / "rna_sequence_network.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, {nx.number_connected_components(G)} connected components")

## Outputs and interpretation

All figures and CSV tables are saved to `outputs/`. The FASTA input is never modified.

**Caveats:** (1) input header abundance must be the last field; (2) terminal gaps are excluded from distance calculation; (3) differences in aligned gap runs are counted position-by-position; (4) graph edges indicate sequence similarity, not direct ancestry; (5) cluster IDs are arbitrary labels.

## 9. Create Clustered Alignment Figure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# PREPARE SEQUENCES

n_sequences = len(alignment)
n_positions = alignment.get_alignment_length()

# Sort sequences by cluster, then abundance (highest first)
order = sorted(
    range(n_sequences),
    key=lambda i: (
        int(clusters[i]),
        -node_abundances[i]
    )
)

# Nucleotide colour scheme
nucleotide_colors = {
    "A": "#66C2A5",
    "C": "#8DA0CB",
    "G": "#FC8D62",
    "T": "#FFD92F",
    "U": "#FFD92F",
    "-": "#EEEEEE",
    "N": "#FFFFFF"
}

nucleotide_codes = {
    "-": 0,
    "A": 1,
    "C": 2,
    "G": 3,
    "T": 4,
    "U": 4,
    "N": 5
}

cmap = ListedColormap([
    nucleotide_colors["-"],
    nucleotide_colors["A"],
    nucleotide_colors["C"],
    nucleotide_colors["G"],
    nucleotide_colors["T"],
    nucleotide_colors["N"]
])

# Convert aligned sequences into numeric matrix
matrix = np.array([
    [
        nucleotide_codes.get(base.upper(), 5)
        for base in str(alignment[i].seq)
    ]
    for i in order
])

# CREATE ALIGNMENT FIGURE

fig, ax = plt.subplots(
    figsize=(20, max(8, n_sequences * 0.35))
)

ax.imshow(
    matrix,
    cmap=cmap,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=5
)

# LABEL SEQUENCES

sequence_labels = [
    f"{alignment[i].id.split('/')[0]}  "
    f"({node_abundances[i]:.2%})"
    for i in order
]

ax.set_yticks(range(n_sequences))
ax.set_yticklabels(sequence_labels, fontsize=9)

# Show alignment positions
ax.set_xticks(range(0, n_positions, 5))
ax.set_xticklabels(
    range(1, n_positions + 1, 5),
    fontsize=10
)

ax.set_xlabel("Aligned nucleotide position", fontsize=12)
ax.set_ylabel("Sequences (abundance)", fontsize=12)

# ADD CLUSTER BOUNDARIES

ordered_clusters = [int(clusters[i]) for i in order]

for row in range(1, n_sequences):
    if ordered_clusters[row] != ordered_clusters[row - 1]:
        ax.axhline(
            row - 0.5,
            color="black",
            linewidth=2
        )

# Label each cluster on the right
for cluster in sorted(set(ordered_clusters)):

    rows = [
        j for j, c in enumerate(ordered_clusters)
        if c == cluster
    ]

    midpoint = np.mean(rows)

    ax.text(
        n_positions + 0.5,
        midpoint,
        f"Cluster {cluster}",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

# ADD NUCLEOTIDE LEGEND

legend_elements = [
    Patch(facecolor=nucleotide_colors[base], label=base)
    for base in ["A", "C", "G", "U", "-"]
]

ax.legend(
    handles=legend_elements,
    title="Nucleotide",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.04),
    ncol=5,
    frameon=False
)

ax.set_title(
    "Multiple sequence alignment\n"
    "Grouped by sequence cluster and ranked by abundance",
    fontsize=16,
    pad=20
)

plt.tight_layout()
plt.show()